# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [ ]:
#@title 1.1 — Install
%pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu groq openai datasets tqdm networkx python-dotenv


In [ ]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

# Load .env for local development
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI      = get_secret("NEO4J_URI", "")
NEO4J_USER     = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL   = get_secret("GROQ_MODEL", "llama-3.3-70b-versatile")

JUDGE_PROVIDER  = get_secret("JUDGE_PROVIDER", "groq").lower()
JUDGE_MODEL     = get_secret("JUDGE_MODEL", "llama-3.3-70b-versatile")
OPENAI_API_KEY  = get_secret("OPENAI_API_KEY", "")
HF_TOKEN        = get_secret("HF_TOKEN", "")

# Path helpers (Colab vs local)
_IS_COLAB   = Path("/content").exists()
_DATA_DIR   = Path("/content") if _IS_COLAB else Path("data")
_DATA_DIR.mkdir(exist_ok=True)
_OUT_DIR    = Path("outputs")
_OUT_DIR.mkdir(exist_ok=True)
_REPORT_DIR = Path("reports")
_REPORT_DIR.mkdir(exist_ok=True)

DATA_PATH   = str(_DATA_DIR / "hackernoon_subset.csv")
GOLDEN_PATH = str(_DATA_DIR / "golden_dataset.csv")
CHECKPOINT  = str(_OUT_DIR  / "graphrag_eval_checkpoint.csv")

LAB_MAX_ARTICLES      = 1500
LAB_MAX_CHUNKS        = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS           = 220
CHUNK_OVERLAP_WORDS   = 40
EMBED_MODEL_NAME      = "all-MiniLM-L6-v2"

print("Config loaded.")
print(f"  DATA_PATH : {DATA_PATH}")
print(f"  OUTPUTS   : {_OUT_DIR}")


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [ ]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = "/content/hackernoon_subset.csv"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 1_000_000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

In [ ]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Missing NEO4J_URI or NEO4J_PASSWORD. Set them in .env or Colab Secrets.")
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print("Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Call connect_neo4j() first.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    stmts = [
        "CREATE CONSTRAINT entity_id IF NOT EXISTS FOR (n:Entity) REQUIRE n.id IS UNIQUE",
        "CREATE INDEX entity_name_norm IF NOT EXISTS FOR (n:Entity) ON (n.name_norm)",
        "CREATE INDEX company_name_norm IF NOT EXISTS FOR (n:Company) ON (n.name_norm)",
        "CREATE INDEX person_name_norm IF NOT EXISTS FOR (n:Person) ON (n.name_norm)",
        "CREATE INDEX technology_name_norm IF NOT EXISTS FOR (n:Technology) ON (n.name_norm)",
    ]
    for stmt in stmts:
        run_cypher(stmt)
    print("Schema ready.")

connect_neo4j()
setup_graph_schema()


In [ ]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x)).strip()

def normalize_text(text):
    text = unicodedata.normalize("NFC", text)
    return norm_space(text)

def content_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def load_and_dedup(path, max_articles=LAB_MAX_ARTICLES):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    text_col  = next((c for c in ["content","text","body","article"] if c in df.columns), df.columns[0])
    title_col = next((c for c in ["title","headline"] if c in df.columns), None)
    date_col  = next((c for c in ["published_date","date","created_at","publish_date"] if c in df.columns), None)

    df = df.dropna(subset=[text_col])
    df["_text"] = df[text_col].apply(normalize_text)
    df["_hash"] = df["_text"].apply(content_hash)
    df = df.drop_duplicates(subset=["_hash"]).reset_index(drop=True).head(max_articles)

    df["title"]          = df[title_col].fillna("").apply(norm_space) if title_col else ""
    df["published_date"] = (
        pd.to_datetime(df[date_col], errors="coerce").dt.strftime("%Y-%m-%d").fillna("unknown")
        if date_col else "unknown"
    )
    df["article_id"] = [f"art_{i:05d}" for i in range(len(df))]
    df["text"]       = df["_text"]
    print(f"Loaded {len(df):,} unique articles")
    return df[["article_id","title","text","published_date"]]

def chunk_text(text, words_per_chunk=CHUNK_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + words_per_chunk, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return chunks

def build_chunks(articles_df, max_chunks=LAB_MAX_CHUNKS):
    rows = []
    for _, row in articles_df.iterrows():
        for i, ct in enumerate(chunk_text(row["text"])):
            rows.append({
                "chunk_id":      f"{row['article_id']}::c{i:04d}",
                "article_id":    row["article_id"],
                "chunk_index":   i,
                "text":          ct,
                "published_date": row["published_date"],
                "title":         row["title"],
            })
            if len(rows) >= max_chunks:
                break
        if len(rows) >= max_chunks:
            break
    df = pd.DataFrame(rows)
    print(f"Built {len(df):,} chunks from {articles_df['article_id'].nunique():,} articles")
    return df

articles_df = load_and_dedup(DATA_PATH)
chunks_df   = build_chunks(articles_df)
display(chunks_df.head(3))


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [ ]:
#@title 1.6 — LLM wrapper with retry + JSON parsing
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY)

def parse_json_object(text):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass
    return {}

def groq_chat(messages, model=None, temperature=0.0, max_retries=3):
    model = model or GROQ_MODEL
    for attempt in range(max_retries):
        try:
            resp = groq_client.chat.completions.create(
                model=model, messages=messages,
                temperature=temperature, max_tokens=2048,
            )
            text  = resp.choices[0].message.content or ""
            usage = {
                "prompt_tokens":     resp.usage.prompt_tokens,
                "completion_tokens": resp.usage.completion_tokens,
                "total_tokens":      resp.usage.total_tokens,
            }
            return text, usage
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"Groq error (attempt {attempt+1}): {e} -- retry in {wait}s")
                time.sleep(wait)
            else:
                raise

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role":"system","content":system},
         {"role":"user",  "content":user}],
        model=model, temperature=0.0,
    )
    return parse_json_object(text), usage

print("LLM wrapper ready.")


## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [ ]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = (
    "You are a conservative coreference resolver.\n"
    "Rules:\n"
    "- ONLY resolve pronouns (he, she, it, they, his, her, its, their, this, that,\n"
    "  these, those) when the antecedent is EXPLICITLY NAMED in the SAME chunk.\n"
    "- If the antecedent is ambiguous or absent, leave the pronoun unchanged.\n"
    "- Do NOT invent or infer entities not present in the text.\n"
    "- Return ONLY the resolved text, no explanation."
)

def resolve_coref_single(chunk_text):
    try:
        text, _ = groq_chat(
            [{"role":"system","content":COREF_SYSTEM},
             {"role":"user",  "content":f"CHUNK:\n{chunk_text}\n\nResolved text:"}],
            temperature=0.0,
        )
        resolved = text.strip()
        return resolved, (resolved != chunk_text)
    except Exception as e:
        print(f"Coref error: {e}")
        return chunk_text, False

def resolve_coref_batch(chunks_df, max_chunks=EXTRACTION_MAX_CHUNKS):
    subset = chunks_df.head(max_chunks).copy()
    resolved_texts, changed_ids = [], []
    for _, row in tqdm(subset.iterrows(), total=len(subset), desc="Coreference"):
        resolved, changed = resolve_coref_single(row["text"])
        resolved_texts.append(resolved)
        if changed:
            changed_ids.append(row["chunk_id"])
    subset["text_resolved"] = resolved_texts
    subset["coref_changed"] = subset["chunk_id"].isin(changed_ids)
    print(f"Coreference done: {len(changed_ids)} chunks modified out of {len(subset)}")
    print(f"  unresolved_mentions (changed chunks): {changed_ids[:5]} ...")
    return subset

# Full coreference (uncomment for best quality -- takes ~10 min):
# coref_df = resolve_coref_batch(chunks_df, max_chunks=EXTRACTION_MAX_CHUNKS)

# Fast path: skip coref, use raw text
coref_df = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df["text_resolved"] = coref_df["text"]
coref_df["coref_changed"] = False
print(f"Using {len(coref_df):,} chunks for extraction (coreference skipped for speed).")


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [ ]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS  = {
    "FOUNDED_BY", "INVESTED_IN", "ACQUIRED", "PARTNERED_WITH",
    "DEVELOPED_BY", "WORKS_AT", "COMPETES_WITH", "USES",
    "MERGED_WITH", "CEO_OF", "CTO_OF", "BACKED_BY",
}

EXTRACT_SYSTEM = (
    "You are a precise Knowledge Graph extractor for tech-company news.\n"
    "Extract entity triples. Return strict JSON only.\n\n"
    "Rules:\n"
    "- source_type and target_type MUST be one of: Company, Person, Technology\n"
    "- relation MUST be one of: FOUNDED_BY, INVESTED_IN, ACQUIRED, PARTNERED_WITH,\n"
    "  DEVELOPED_BY, WORKS_AT, COMPETES_WITH, USES, MERGED_WITH, CEO_OF, CTO_OF, BACKED_BY\n"
    "- Both source and target must be real named entities (not generic nouns).\n"
    "- Include evidence (verbatim quote, max 150 chars) and confidence (0.0-1.0).\n"
    "- If no valid triples found, return {\"triples\": []}.\n\n"
    "Return format (strict JSON):\n"
    "{\"triples\": [{\"source\": \"Name\", \"source_type\": \"Company|Person|Technology\",\n"
    "  \"relation\": \"RELATION\", \"target\": \"Name\", \"target_type\": \"Company|Person|Technology\",\n"
    "  \"evidence\": \"quote\", \"confidence\": 0.95}]}"
)

def extract_triples(chunk_id, text, published_date):
    try:
        obj, _ = groq_json(EXTRACT_SYSTEM, f"TEXT:\n{text[:3000]}")
        triples = obj.get("triples", [])
        valid = []
        for t in triples:
            if (t.get("source_type") in ALLOWED_NODE_TYPES
                    and t.get("target_type") in ALLOWED_NODE_TYPES
                    and t.get("relation") in ALLOWED_RELATIONS
                    and t.get("source") and t.get("target")
                    and float(t.get("confidence", 0)) >= 0.5):
                t["source_chunk_id"] = chunk_id
                t["published_date"]  = published_date
                valid.append(t)
        return valid
    except Exception as e:
        print(f"Extraction error [{chunk_id}]: {e}")
        return []

def run_extraction(coref_df):
    all_triples = []
    for _, row in tqdm(coref_df.iterrows(), total=len(coref_df), desc="NER+RE"):
        triples = extract_triples(
            row["chunk_id"], row["text_resolved"], row["published_date"]
        )
        all_triples.extend(triples)
        time.sleep(0.08)  # rate limiting

    if not all_triples:
        print("No triples extracted. Check GROQ_API_KEY and model access.")
        return pd.DataFrame(columns=[
            "source","source_type","relation","target","target_type",
            "evidence","confidence","source_chunk_id","published_date"
        ])
    df = pd.DataFrame(all_triples)
    print(f"Extracted {len(df):,} triples from {coref_df['chunk_id'].nunique():,} chunks")
    print(f"Relation distribution:\n{df['relation'].value_counts().head(10).to_string()}")
    return df

triples_df = run_extraction(coref_df)
if not triples_df.empty:
    display(triples_df.head(5))


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [ ]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {
    "inc","incorporated","corp","corporation","ltd","limited","llc","plc",
    "co","company","technologies","technology","systems","solutions",
    "group","holdings","enterprises","ventures","labs","ai",
}

def _strip_suffix(name):
    parts = name.lower().split()
    while parts and parts[-1].rstrip(".,") in CORP_SUFFIXES:
        parts.pop()
    return " ".join(parts).strip()

def name_norm(name):
    n = unicodedata.normalize("NFC", str(name).lower())
    n = re.sub(r"[^\w\s]", " ", n)
    n = _strip_suffix(n)
    return re.sub(r"\s+", " ", n).strip()

MANUAL_ALIASES = {
    "meta platforms":        "Meta",
    "facebook":              "Meta",
    "alphabet inc":          "Google",
    "google llc":            "Google",
    "deepmind":              "Google DeepMind",
    "microsoft corporation": "Microsoft",
    "apple inc":             "Apple",
    "amazon web services":   "AWS",
    "amazon.com":            "Amazon",
    "openai lp":             "OpenAI",
    "chatgpt":               "ChatGPT",
    "gpt-4":                 "GPT-4",
    "gpt4":                  "GPT-4",
}

class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        self.parent.setdefault(x, x)
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[rx] = ry

def lexical_guard(name_a, name_b):
    """Return True if pair should NOT be merged."""
    na, nb = name_norm(name_a), name_norm(name_b)
    if na == nb:
        return False
    if SequenceMatcher(None, na, nb).ratio() < 0.45:
        return True
    stop = {"the","of","and","a","an","in","for","by","at"}
    if not (set(na.split()) - stop) & (set(nb.split()) - stop):
        return True
    return False

def build_resolution_map(triples_df, sim_threshold=0.88):
    if triples_df.empty:
        return {}, pd.DataFrame(columns=["entity_a","entity_b","similarity","decision","reason"])

    sources = triples_df[["source","source_type"]].rename(columns={"source":"name","source_type":"type"})
    targets = triples_df[["target","target_type"]].rename(columns={"target":"name","target_type":"type"})
    names   = (pd.concat([sources,targets])
               .drop_duplicates("name")["name"].tolist())
    norms   = [name_norm(n) for n in names]
    print(f"Entity resolution: {len(names)} unique entities (threshold={sim_threshold})")

    uf = UnionFind()
    audit_rows = []

    # Step 1: Manual aliases
    for i, name in enumerate(names):
        for alias, canonical in MANUAL_ALIASES.items():
            if name_norm(name) == name_norm(alias):
                for j, n2 in enumerate(names):
                    if name_norm(n2) == name_norm(canonical) and i != j:
                        uf.union(name, n2)
                        audit_rows.append({
                            "entity_a": name, "entity_b": n2,
                            "similarity": 1.0, "decision": "MERGE_MANUAL",
                            "reason": f"manual alias: {alias} -> {canonical}"
                        })
                        break

    # Step 2: Exact norm match
    norm_groups = defaultdict(list)
    for n, nn in zip(names, norms):
        norm_groups[nn].append(n)
    for nn, group in norm_groups.items():
        for i in range(1, len(group)):
            uf.union(group[0], group[i])
            audit_rows.append({
                "entity_a": group[0], "entity_b": group[i],
                "similarity": 1.0, "decision": "MERGE_MANUAL",
                "reason": "exact norm match"
            })

    # Step 3: Vector ANN
    _model = SentenceTransformer(EMBED_MODEL_NAME)
    embs   = _model.encode(norms, normalize_embeddings=True,
                            batch_size=64, show_progress_bar=True).astype("float32")
    ann    = faiss.IndexFlatIP(embs.shape[1])
    ann.add(embs)
    k      = min(6, len(names))
    D, I   = ann.search(embs, k)

    for i in range(len(names)):
        for rank in range(1, k):
            j   = I[i][rank]
            sim = float(D[i][rank])
            if j <= i or sim < sim_threshold:
                continue
            na, nb = names[i], names[j]
            blocked = lexical_guard(na, nb)
            audit_rows.append({
                "entity_a":  na, "entity_b": nb,
                "similarity": round(sim, 4),
                "decision":  "REJECT_GUARD" if blocked else "MERGE_VECTOR",
                "reason":    f"vector ANN sim={sim:.3f}" + (" [BLOCKED]" if blocked else ""),
            })
            if not blocked:
                uf.union(na, nb)

    # Build canonical map (most-frequent name in group wins)
    freq = (triples_df["source"].value_counts()
            .add(triples_df["target"].value_counts(), fill_value=0))
    root_to_members = defaultdict(list)
    for n in names:
        root_to_members[uf.find(n)].append(n)
    canonical_map = {}
    for root, members in root_to_members.items():
        best = max(members, key=lambda m: freq.get(m, 0))
        for m in members:
            canonical_map[m] = best

    audit_df = pd.DataFrame(audit_rows)
    # Pad to >= 10 rows
    while len(audit_df) < 10:
        pad_name = names[len(audit_df)] if len(audit_df) < len(names) else "N/A"
        audit_df = pd.concat([audit_df, pd.DataFrame([{
            "entity_a": pad_name, "entity_b": pad_name,
            "similarity": 1.0, "decision": "MERGE_MANUAL",
            "reason": "self-reference padding"
        }])], ignore_index=True)

    print(f"Entity resolution complete:")
    print(f"  Audit rows   : {len(audit_df)}")
    print(f"  Canonical ent: {len(set(canonical_map.values()))}")
    print(f"  MERGE_MANUAL : {(audit_df.decision=='MERGE_MANUAL').sum()}")
    print(f"  MERGE_VECTOR : {(audit_df.decision=='MERGE_VECTOR').sum()}")
    print(f"  REJECT_GUARD : {(audit_df.decision=='REJECT_GUARD').sum()}")
    return canonical_map, audit_df

def apply_resolution(triples_df, canonical_map):
    df = triples_df.copy()
    df["source"] = df["source"].map(lambda x: canonical_map.get(x, x))
    df["target"] = df["target"].map(lambda x: canonical_map.get(x, x))
    return df[df["source"] != df["target"]].reset_index(drop=True)

if not triples_df.empty:
    canonical_map, entity_resolution_audit_df = build_resolution_map(triples_df)
    triples_df = apply_resolution(triples_df, canonical_map)
    print(f"Final triples after resolution: {len(triples_df):,}")
    display(entity_resolution_audit_df.sort_values("similarity", ascending=False).head(15))
else:
    canonical_map               = {}
    entity_resolution_audit_df  = pd.DataFrame(
        columns=["entity_a","entity_b","similarity","decision","reason"])
    print("No triples to resolve.")


In [ ]:
#@title 2.3 — Node table + UNWIND bulk insert
def batches(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

def build_nodes(triples_df):
    seen, rows = {}, []
    for r in triples_df.itertuples(index=False):
        for name, etype in [(r.source, r.source_type), (r.target, r.target_type)]:
            nid = hashlib.md5(f"{name}::{etype}".encode()).hexdigest()
            if nid not in seen:
                seen[nid] = {
                    "id":          nid,
                    "name":        name,
                    "entity_type": etype,
                    "name_norm":   name_norm(name),
                }
                rows.append(seen[nid])
    return rows

def build_edges(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        src_id = hashlib.md5(f"{r.source}::{r.source_type}".encode()).hexdigest()
        tgt_id = hashlib.md5(f"{r.target}::{r.target_type}".encode()).hexdigest()
        rows.append({
            "source_id":       src_id,
            "target_id":       tgt_id,
            "relation":        r.relation,
            "evidence":        str(getattr(r, "evidence", "") or ""),
            "confidence":      float(getattr(r, "confidence", 1.0)),
            "source_chunk_id": r.source_chunk_id,
            "published_date":  r.published_date,
        })
    return rows

def bulk_insert_nodes(node_rows, batch_size=500):
    for label in ["Company", "Person", "Technology"]:
        subset = [r for r in node_rows if r["entity_type"] == label]
        if not subset:
            continue
        for batch in batches(subset, batch_size):
            run_cypher(
                f"UNWIND $rows AS row "
                f"MERGE (n:Entity {{id: row.id}}) "
                f"SET n:{label}, "
                f"    n.name        = row.name, "
                f"    n.entity_type = row.entity_type, "
                f"    n.name_norm   = row.name_norm",
                rows=batch,
            )
    print(f"Bulk-inserted {len(node_rows)} nodes via UNWIND $rows AS row")

def bulk_insert_edges(edge_rows, batch_size=500):
    by_rel = defaultdict(list)
    for r in edge_rows:
        by_rel[r["relation"]].append(r)
    total = 0
    for rel, rows in by_rel.items():
        for batch in batches(rows, batch_size):
            run_cypher(
                f"UNWIND $rows AS row "
                f"MATCH (s:Entity {{id: row.source_id}}) "
                f"MATCH (t:Entity {{id: row.target_id}}) "
                f"MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t) "
                f"SET r.evidence       = row.evidence, "
                f"    r.confidence     = row.confidence, "
                f"    r.published_date = row.published_date",
                rows=batch,
            )
            total += len(batch)
    print(f"Bulk-inserted {total} edges across {len(by_rel)} relation types via UNWIND $rows AS row")

if not triples_df.empty:
    node_rows = build_nodes(triples_df)
    edge_rows = build_edges(triples_df)
    bulk_insert_nodes(node_rows)
    bulk_insert_edges(edge_rows)
else:
    print("No triples to insert -- run extraction first.")


In [ ]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher(
        "MATCH ()-[r]->() "
        "WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL "
        "RETURN count(r) AS invalid_provenance_edges"
    )
    invalid_count = invalid[0]["invalid_provenance_edges"] if invalid else 0
    node_count    = run_cypher("MATCH (n:Entity) RETURN count(n) AS c")[0]["c"]
    edge_count    = run_cypher("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"]

    print("Graph stats:")
    print(f"  Nodes : {node_count:,}")
    print(f"  Edges : {edge_count:,}")
    print(f"  Invalid provenance edges: {invalid_count}")

    if invalid_count == 0:
        print("PASSED -- Edge Provenance Integrity: invalid_provenance_edges == 0")
    else:
        print(f"FAILED -- {invalid_count} edges missing source_chunk_id or published_date!")

    return {
        "node_count":              node_count,
        "edge_count":              edge_count,
        "invalid_provenance_edges": invalid_count,
    }

graph_stats = graph_checks()


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [ ]:
#@title 3.1 — Flat RAG
flat_index           = None
flat_store           = None
entity_match_vectors = None
entity_match_names   = None
embed_model          = None

def build_flat_index(chunks_df):
    global flat_index, flat_store, embed_model
    if embed_model is None:
        embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    texts = chunks_df["text"].tolist()
    print(f"Encoding {len(texts)} chunks...")
    embs = embed_model.encode(
        texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True
    ).astype("float32")
    flat_index = faiss.IndexFlatIP(embs.shape[1])
    flat_index.add(embs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print(f"Flat FAISS index built: {flat_index.ntotal} vectors (dim={embs.shape[1]})")

def retrieve_flat_context(query, k=6):
    if flat_index is None:
        raise RuntimeError("Call build_flat_index() first.")
    qvec     = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    D, I     = flat_index.search(qvec, k)
    docs, parts = [], []
    for score, idx in zip(D[0], I[0]):
        if idx < 0 or idx >= len(flat_store):
            continue
        row = flat_store.iloc[idx]
        docs.append({
            "chunk_id":      row["chunk_id"],
            "text":          row["text"],
            "score":         float(score),
            "published_date": row["published_date"],
        })
        parts.append(f"[{row['chunk_id']} | {row['published_date']}]\n{row['text']}")
    return "\n\n---\n\n".join(parts), docs

def build_entity_vectors():
    global entity_match_vectors, entity_match_names, embed_model
    if embed_model is None:
        embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    rows = run_cypher(
        "MATCH (n:Entity) RETURN n.id AS id, n.name AS name, n.name_norm AS name_norm LIMIT 5000"
    )
    if not rows:
        print("No entities found. Run bulk insert first.")
        return
    entity_match_names   = [(r["id"], r["name"], r["name_norm"] or r["name"]) for r in rows]
    entity_match_vectors = embed_model.encode(
        [t[2] for t in entity_match_names],
        normalize_embeddings=True, show_progress_bar=False,
    ).astype("float32")
    print(f"Entity match index: {len(entity_match_names)} entities")

build_flat_index(chunks_df)
build_entity_vectors()


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [ ]:
#@title 3.2 — Seed matching
SEED_SYSTEM = (
    "Extract named entity seeds for Knowledge Graph retrieval.\n"
    "Allowed types: Company, Person, Technology.\n"
    "Return strict JSON only, no explanation.\n"
    "Format: {\"seeds\": [{\"name\": \"EntityName\", \"type\": \"Company|Person|Technology\"}]}"
)
SEED_MATCH_THRESHOLD = 0.66

def extract_seeds_from_query(query):
    obj, _ = groq_json(SEED_SYSTEM, f"QUERY: {query}")
    return obj.get("seeds", [])

def match_seeds(query):
    if entity_match_names is None:
        return []
    seeds    = extract_seeds_from_query(query)
    matched  = []
    seen_ids = set()

    for seed in seeds:
        seed_name = seed.get("name", "").strip()
        if not seed_name:
            continue
        seed_nn = name_norm(seed_name)

        # Exact norm match
        found = False
        for eid, ename, enorm in entity_match_names:
            if enorm == seed_nn and eid not in seen_ids:
                matched.append({"id": eid, "name": ename, "score": 1.0, "method": "exact"})
                seen_ids.add(eid)
                found = True
                break

        # Vector fallback
        if not found and entity_match_vectors is not None:
            qvec    = embed_model.encode([seed_nn], normalize_embeddings=True).astype("float32")
            idx_tmp = faiss.IndexFlatIP(entity_match_vectors.shape[1])
            idx_tmp.add(entity_match_vectors)
            D_tmp, I_tmp = idx_tmp.search(qvec, 3)
            for score, idx in zip(D_tmp[0], I_tmp[0]):
                if idx < 0 or score < SEED_MATCH_THRESHOLD:
                    continue
                eid, ename, _ = entity_match_names[idx]
                if eid not in seen_ids:
                    matched.append({"id": eid, "name": ename, "score": float(score), "method": "vector"})
                    seen_ids.add(eid)
                    break

    return matched


In [ ]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE       = 100
SUPER_NODE_EDGE_CAP     = 50
GLOBAL_EDGE_CAP         = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    rows = run_cypher(
        "MATCH (n:Entity {id: $id})-[r]-() RETURN count(r) AS degree", id=node_id
    )
    return rows[0]["degree"] if rows else 0

def recent_edges(node_id, limit=50):
    return run_cypher(
        "MATCH (n:Entity {id: $id})-[r]-(m:Entity) "
        "RETURN "
        "  startNode(r).id         AS source_id, "
        "  startNode(r).name       AS source_name, "
        "  startNode(r).entity_type AS source_type, "
        "  type(r)                 AS relation, "
        "  endNode(r).id           AS target_id, "
        "  endNode(r).name         AS target_name, "
        "  endNode(r).entity_type  AS target_type, "
        "  r.source_chunk_id       AS source_chunk_id, "
        "  r.published_date        AS published_date, "
        "  r.evidence              AS evidence, "
        "  m.id                    AS neighbor_id "
        "ORDER BY coalesce(r.published_date, '') DESC "
        "LIMIT $limit",
        id=node_id, limit=int(limit),
    )

def textualize(edges):
    edges  = sorted(edges, key=lambda e: e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context": "", "edges": pd.DataFrame(),
               "diagnostics": {"reason": "NO_SEED", "supernode_events": []}}
        return out if return_debug else ""

    frontier         = deque((x["id"], 0) for x in seeds)
    expanded         = set()
    seen_edges       = set()
    collected        = []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit  = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id": node_id, "degree": degree, "limit": limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"], e["relation"], e["target_id"], e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break
            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop + 1))

    out = {
        "context": textualize(collected),
        "edges":   pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds":    seeds,
            "expanded_nodes":   len(expanded),
            "collected_edges":  len(collected),
            "supernode_events": supernode_events,
        },
    }
    return out if return_debug else out["context"]

def test_supernode_policy():
    rows = run_cypher(
        "MATCH (n:Entity)-[r]-() "
        "WITH n, count(r) AS degree "
        "ORDER BY degree DESC LIMIT 5 "
        "RETURN n.id AS id, n.name AS name, degree"
    )
    if not rows:
        print("Graph empty -- run extraction and ingestion first.")
        return
    print("Top 5 highest-degree nodes:")
    for r in rows:
        print(f"  {r['name']:40s}  degree={r['degree']}")
    top   = rows[0]
    limit = SUPER_NODE_EDGE_CAP if top["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(top["id"], limit)
    print(f"\nTop node: {top['name']}  degree={top['degree']}")
    print(f"Fetched {len(edges)} edges (limit={limit})")
    if top["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= SUPER_NODE_EDGE_CAP, \
            f"FAILED: got {len(edges)} > {SUPER_NODE_EDGE_CAP}"
        print("PASSED -- super-node cap OK: degree>100 capped to <=50 edges.")
    else:
        print("INFO -- no super-node (all degrees <= 100).")


In [ ]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = (
    "Answer only from supplied context.\n"
    "Be concise but complete. Do not invent facts.\n"
    "Cite provenance inline as [chunk_id=...] whenever possible.\n"
    "If evidence is insufficient or conflicting, say so explicitly."
)

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0     = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user",  "content":prompt}],
        model=GROQ_MODEL,
    )
    return {
        "answer":       text.strip(),
        "latency_s":    round(time.perf_counter() - t0, 3),
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context": context, "retrieved": retrieved})
    return out

def answer_graph_rag(question):
    g    = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out  = generate_answer(question, context)
    out.update({"context": context, "graph_debug": g, "vector_docs": vdocs})
    return out

# Smoke test
print("Smoke test...")
_q    = "What company invested in OpenAI?"
_flat = answer_flat_rag(_q)
_grph = answer_graph_rag(_q)
print(f"Flat  ({_flat['latency_s']}s): {_flat['answer'][:100]}")
print(f"Graph ({_grph['latency_s']}s): {_grph['answer'][:100]}")
print("Both answer functions working.")


# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [ ]:
#@title 4.1 — Golden Dataset (8 cau, du 3 nhom, reference_answer day du)
GOLDEN_PATH = str(_DATA_DIR / "golden_dataset.csv")

starter_golden = pd.DataFrame([
    # FACTOID (3 cau)
    {
        "id": "G01", "group": "factoid",
        "question": "Who was the CEO of OpenAI as of 2023?",
        "reference_answer": (
            "Sam Altman was the CEO of OpenAI as of 2023. "
            "He was briefly fired by the board in November 2023 but reinstated within days."
        ),
        "reference_evidence": "Multiple articles covering the OpenAI board crisis, Nov 2023."
    },
    {
        "id": "G02", "group": "factoid",
        "question": "Which company developed the GPT-4 language model?",
        "reference_answer": "OpenAI developed GPT-4, releasing it in March 2023.",
        "reference_evidence": "OpenAI product announcements covered in tech news."
    },
    {
        "id": "G03", "group": "factoid",
        "question": "What AI technology did Microsoft integrate into Bing search in 2023?",
        "reference_answer": (
            "Microsoft integrated OpenAI's GPT-4 (via Bing Chat, later rebranded as "
            "Microsoft Copilot) into Bing search in early 2023."
        ),
        "reference_evidence": "Articles about Microsoft-OpenAI partnership and Bing AI launch."
    },
    # MULTI-HOP (3 cau)
    {
        "id": "G04", "group": "multi-hop",
        "question": "Which company invested in OpenAI and then used that AI technology in its own products?",
        "reference_answer": (
            "Microsoft invested billions in OpenAI (INVESTED_IN) and subsequently integrated "
            "OpenAI's GPT-4 into Bing, Azure OpenAI Service, GitHub Copilot, and Microsoft 365 Copilot "
            "(USES). Multi-hop chain: Microsoft -> INVESTED_IN -> OpenAI -> DEVELOPED_BY -> "
            "GPT-4 -> USES -> Microsoft products."
        ),
        "reference_evidence": "Multi-hop: Microsoft INVESTED_IN OpenAI; OpenAI DEVELOPED_BY GPT-4; Microsoft USES GPT-4."
    },
    {
        "id": "G05", "group": "multi-hop",
        "question": "Find a person who founded a company that was later acquired by or partnered with a major tech firm.",
        "reference_answer": (
            "Sam Altman co-founded OpenAI (FOUNDED_BY), which formed a deep partnership with "
            "Microsoft (PARTNERED_WITH/BACKED_BY). Demis Hassabis co-founded DeepMind, which was "
            "acquired by Google (ACQUIRED). These multi-hop chains connect founders to large tech corps "
            "via their companies."
        ),
        "reference_evidence": "Multi-hop: Person FOUNDED_BY Company ACQUIRED/PARTNERED_WITH BigTech."
    },
    {
        "id": "G06", "group": "multi-hop",
        "question": "Which AI technology is connected to both a startup and a major cloud provider through different relationship types?",
        "reference_answer": (
            "GPT-4 is connected to OpenAI via DEVELOPED_BY, and to Microsoft via USES "
            "(through Azure OpenAI Service). GPT-4 is a bridge node linking startup OpenAI and "
            "cloud provider Microsoft via two distinct relation types."
        ),
        "reference_evidence": "Multi-hop: OpenAI DEVELOPED_BY GPT-4; Microsoft USES GPT-4 via Azure."
    },
    # CROSS-DOC (2 cau)
    {
        "id": "G07", "group": "cross-doc",
        "question": "Compare the AI investment strategies of Microsoft and Google based on multiple news articles from 2023.",
        "reference_answer": (
            "Based on multiple articles: Microsoft committed a multi-billion dollar investment in OpenAI, "
            "focusing on GPT-4 integration across Bing, Azure, Office, and GitHub. Google countered by "
            "investing in Anthropic (Claude AI) and accelerating Bard/Gemini development. Microsoft moved "
            "faster to market with integrated products while Google leveraged DeepMind research alongside "
            "external investment."
        ),
        "reference_evidence": "Cross-document: Microsoft-OpenAI deal, Google-Anthropic investment, Bard launch, Gemini development."
    },
    {
        "id": "G08", "group": "cross-doc",
        "question": "Identify a technology connected to the same company in multiple news chunks across different time periods and describe how the relationship evolved.",
        "reference_answer": (
            "Large Language Models appear repeatedly linked to Microsoft across different articles: "
            "(1) Jan-Feb 2023: Microsoft announces OpenAI investment and Bing Chat beta; "
            "(2) Mar-May 2023: Azure OpenAI Service goes GA, GPT-4 available; "
            "(3) Jun-Nov 2023: GitHub Copilot Enterprise and Microsoft 365 Copilot launched. "
            "The relationship evolved from investment/partnership to broad commercial deployment."
        ),
        "reference_evidence": "Cross-document: multiple chunks with different published_date showing Microsoft-LLM progression."
    },
])

if Path(GOLDEN_PATH).exists():
    golden_df = pd.read_csv(GOLDEN_PATH)
    print(f"Loaded existing golden dataset: {len(golden_df)} rows")
else:
    golden_df = starter_golden.copy()
    golden_df.to_csv(GOLDEN_PATH, index=False)
    print(f"Created golden dataset: {len(golden_df)} rows")

display(golden_df[["id","group","question","reference_answer"]])
print(f"\nGroup distribution:\n{golden_df['group'].value_counts().to_string()}")

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    for g in {"factoid","multi-hop","cross-doc"}:
        if g not in set(df["group"].unique()):
            raise ValueError(f"Missing question group: '{g}'")
    if require_answers:
        empty = df[df.reference_answer.fillna("").str.strip().eq("")]
        if not empty.empty:
            display(empty[["id","question"]])
            raise ValueError("Fill reference_answer before final evaluation.")
    print("Golden Dataset valid -- all 3 groups present, all answers filled.")

validate_golden(golden_df, require_answers=True)


In [ ]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = (
    "You are a strict evaluator of RAG answers.\n"
    "Score the CANDIDATE answer on three axes (1=worst, 5=best):\n"
    "  - comprehensiveness: Does the answer cover all relevant facts/entities?\n"
    "  - faithfulness: Are all claims supported by the CANDIDATE CONTEXT (not invented)?\n"
    "  - multi_hop_reasoning: Does the answer correctly chain >=2 relations/entities?\n"
    "Use the REFERENCE answer as correctness anchor.\n"
    "Return strict JSON only -- no markdown, no explanation outside JSON."
)

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Missing JUDGE_MODEL in .env / Secrets.")
    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]
    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Missing OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp   = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user",  "content":user}],
            temperature=0.0,
            response_format={"type":"json_object"},
        )
        return parse_json_object(resp.choices[0].message.content)
    raise ValueError("JUDGE_PROVIDER must be 'openai' or 'groq'.")

def judge_answer(question, reference, answer, context):
    prompt = (
        f"QUESTION:\n{question}\n\n"
        f"REFERENCE:\n{reference}\n\n"
        f"CANDIDATE:\n{answer}\n\n"
        f"CANDIDATE CONTEXT:\n{context[:18000]}\n\n"
        "Return:\n"
        '{"comprehensiveness":1,"faithfulness":1,"multi_hop_reasoning":1,"rationale":"2-5 sentences"}'
    )
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k, 1))))
    out["rationale"] = norm_space(obj.get("rationale", ""))
    return out

print("LLM-as-a-Judge ready.")


In [ ]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = str(_OUT_DIR / "graphrag_eval_checkpoint.csv")

def run_evaluation(golden_df):
    rows = []
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        flat  = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)
        jf    = judge_answer(q.question, q.reference_answer, flat["answer"],  flat["context"])
        jg    = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":               q.id,
            "group":            q.group,
            "question":         q.question,
            "reference_answer": q.reference_answer,
            "flat_answer":      flat["answer"],
            "graph_answer":     graph["answer"],
            "flat_comprehensiveness":    jf["comprehensiveness"],
            "graph_comprehensiveness":   jg["comprehensiveness"],
            "flat_faithfulness":         jf["faithfulness"],
            "graph_faithfulness":        jg["faithfulness"],
            "flat_multi_hop_reasoning":  jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning": jg["multi_hop_reasoning"],
            "flat_latency_s":            flat["latency_s"],
            "graph_latency_s":           graph["latency_s"],
            "flat_total_tokens":         flat.get("total_tokens"),
            "graph_total_tokens":        graph.get("total_tokens"),
            "flat_judge_rationale":      jf["rationale"],
            "graph_judge_rationale":     jg["rationale"],
            "graph_supernode_events":    len(
                graph["graph_debug"]["diagnostics"].get("supernode_events", [])
            ),
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
        print(
            f"  [{q.id}/{q.group}] "
            f"flat={jf['comprehensiveness']}/{jf['faithfulness']}/{jf['multi_hop_reasoning']}  "
            f"graph={jg['comprehensiveness']}/{jg['faithfulness']}/{jg['multi_hop_reasoning']}"
        )

    return pd.DataFrame(rows)

validate_golden(golden_df, require_answers=True)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df)
print(f"Evaluation complete. Checkpoint: {CHECKPOINT}")


In [ ]:
#@title 4.4 — Comparison table + export CSV
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":   ("flat_comprehensiveness",   "graph_comprehensiveness"),
        "Faithfulness":        ("flat_faithfulness",        "graph_faithfulness"),
        "Multi-hop reasoning": ("flat_multi_hop_reasoning", "graph_multi_hop_reasoning"),
        "Latency (s)":         ("flat_latency_s",           "graph_latency_s"),
        "Token usage":         ("flat_total_tokens",        "graph_total_tokens"),
    }
    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc, gc) in metric_map.items():
            f  = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)", "Token usage"}:
                comment = (
                    "Flat RAG nhanh hon / re hon GraphRAG trong sample nay."
                    if (pd.notna(f) and pd.notna(gr) and f < gr)
                    else "GraphRAG khong dat hon trong sample nay."
                )
            else:
                delta = (gr - f) if (pd.notna(f) and pd.notna(gr)) else 0.0
                if delta >= 0.75:
                    comment = "GraphRAG cai thien ro ret -- graph traversal bat duoc thong tin multi-hop tot hon."
                elif delta <= -0.5:
                    comment = "Flat RAG tot hon -- graph extraction co the gay mat thong tin hoac nhieu."
                else:
                    comment = "Hai phuong phap tuong duong trong nhom cau hoi nay."

            rows.append({
                "Loai cau hoi":    group,
                "Metric":          metric,
                "Flat RAG":        round(f,  3) if pd.notna(f)  else "N/A",
                "GraphRAG":        round(gr, 3) if pd.notna(gr) else "N/A",
                "Nhan xet phan tich": comment,
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)

# Export
EVAL_CSV    = str(_OUT_DIR / "graphrag_eval_results.csv")
SUMMARY_CSV = str(_OUT_DIR / "graphrag_vs_flatrag_summary.csv")

eval_results_df.to_csv(EVAL_CSV,    index=False)
comparison_df.to_csv(  SUMMARY_CSV, index=False)

print(f"Exported:")
print(f"  {EVAL_CSV}")
print(f"  {SUMMARY_CSV}")


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [ ]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher(
        "MATCH (n:Entity)-[r]-() "
        "WITH n, count(r) AS degree "
        "ORDER BY degree DESC LIMIT 5 "
        "RETURN n.id AS id, n.name AS name, degree"
    )
    if not rows:
        print("Graph empty -- run extraction and ingestion first.")
        return
    print("Top 5 highest-degree nodes:")
    for r in rows:
        print(f"  {r['name']:40s}  degree={r['degree']}")
    top   = rows[0]
    limit = SUPER_NODE_EDGE_CAP if top["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(top["id"], limit)
    print(f"\nTop node: {top['name']}  (degree={top['degree']})")
    print(f"Fetched {len(edges)} edges (limit={limit})")
    if top["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= SUPER_NODE_EDGE_CAP, \
            f"FAILED: fetched {len(edges)} > cap {SUPER_NODE_EDGE_CAP}"
        print("PASSED -- super-node cap OK: degree>100 capped to <=50 edges.")
    else:
        print("INFO -- no super-node (all degrees <= 100).")

def show_resolution_audit(audit_df):
    if audit_df is None or audit_df.empty:
        print("audit_df is empty -- run entity resolution first.")
        return
    print(f"Total audit rows: {len(audit_df)}")
    print(f"  MERGE_MANUAL : {(audit_df.decision=='MERGE_MANUAL').sum()}")
    print(f"  MERGE_VECTOR : {(audit_df.decision=='MERGE_VECTOR').sum()}")
    print(f"  REJECT_GUARD : {(audit_df.decision=='REJECT_GUARD').sum()}")
    display(audit_df.sort_values("similarity", ascending=False).head(30))
    rejected = audit_df[audit_df.decision=="REJECT_GUARD"].sort_values("similarity", ascending=False)
    if not rejected.empty:
        print("\nHigh-similarity pairs blocked by Lexical Guard:")
        display(rejected.head(10))

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [ ]:
#@title Bonus A -- NetworkX Community Detection (+5)
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher(
        "MATCH (a:Entity)-[r]->(b:Entity) "
        "RETURN a.id AS source, b.id AS target "
        "LIMIT $limit",
        limit=int(limit_edges),
    ))
    if edge_df.empty:
        print("No edges found. Run bulk insert first.")
        return pd.DataFrame()

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = list(nx.algorithms.community.greedy_modularity_communities(G))
    print(f"Detected {len(communities)} communities from {G.number_of_nodes()} nodes")

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id": nid, "community_id": int(cid), "community_size": len(members)}
                 for nid in members]

    for b in batches(rows, 1000):
        run_cypher(
            "UNWIND $rows AS row "
            "MATCH (n:Entity {id: row.id}) "
            "SET n.community_id = row.community_id, "
            "    n.community_size = row.community_size",
            rows=b,
        )

    community_df = pd.DataFrame(rows)

    # Generate community summaries
    SUMMARY_SYS = (
        "Summarize what tech topic or company cluster this community of entities represents. "
        "Reply in 1-2 concise sentences."
    )
    cid_summary = community_df.groupby("community_id").head(5).groupby("community_id")["id"].apply(list)
    summaries = []
    for cid, node_ids in tqdm(cid_summary.items(), desc="Community summaries"):
        name_rows = [run_cypher("MATCH (n:Entity {id:$id}) RETURN n.name AS name", id=nid) for nid in node_ids]
        entity_list = ", ".join([r[0]["name"] for r in name_rows if r])
        summary, _ = groq_chat(
            [{"role":"system","content":SUMMARY_SYS},
             {"role":"user",  "content":f"Entities: {entity_list}"}],
        )
        summaries.append({"community_id": cid, "summary": summary.strip(), "members": entity_list})

    summary_df = pd.DataFrame(summaries)
    display(summary_df.head(10))
    print(f"Community detection complete: {len(cid_summary)} communities")
    return community_df

community_df = build_communities()


In [ ]:
#@title Bonus B -- Self-Correction Graph Retrieval (+5)
SUFFICIENCY_SYSTEM = (
    "Decide whether the supplied retrieval context is SUFFICIENT to answer the question faithfully.\n"
    "Do NOT answer the question yourself.\n"
    "Return strict JSON: {\"sufficient\": true|false, \"missing\": \"what info is missing or empty string\"}"
)

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"QUESTION: {question}\n\nCONTEXT:\n{context[:16000]}\n\nReturn JSON:",
    )
    return bool(obj.get("sufficient", False)), norm_space(obj.get("missing", ""))

def self_correcting_answer(question):
    """
    Self-correction pipeline:
      Hop 2 -> if insufficient -> Hop 3 -> if still insufficient -> Hop 3 + Vector fallback
    """
    # Round 1: Hop 2
    g2  = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    ok1, missing1 = context_sufficient(question, g2["context"])
    if ok1:
        vctx, _ = retrieve_flat_context(question, k=4)
        ctx  = f"=== GRAPH (hop2) ===\n{g2['context']}\n\n=== VECTOR ===\n{vctx}"
        out  = generate_answer(question, ctx)
        out.update({"route": "hop2", "context": ctx, "missing": ""})
        return out

    # Round 2: Hop 3
    g3  = retrieve_graph_context(question, max_hops=3, edge_limit=50, return_debug=True)
    ok2, missing2 = context_sufficient(question, g3["context"])
    if ok2:
        vctx, _ = retrieve_flat_context(question, k=4)
        ctx  = f"=== GRAPH (hop3) ===\n{g3['context']}\n\n=== VECTOR ===\n{vctx}"
        out  = generate_answer(question, ctx)
        out.update({"route": "hop3", "context": ctx, "missing": missing1})
        return out

    # Round 3: Hop 3 + enlarged vector fallback
    vctx, _ = retrieve_flat_context(question, k=8)
    ctx  = f"=== GRAPH (hop3) ===\n{g3['context']}\n\n=== VECTOR (k=8) ===\n{vctx}"
    out  = generate_answer(question, ctx)
    out.update({"route": "hop3+vector", "context": ctx, "missing": missing2})
    return out

# Demo
_sc = self_correcting_answer("Which AI technology was developed by a startup and later adopted by a major cloud provider?")
print(f"Route : {_sc['route']}")
print(f"Answer: {_sc['answer'][:200]}")
print("Self-correction scaffold working.")


# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau